# HRB Volatility Seasonality Study

## H&R Block — The Most Seasonal Stock in the Market?

**The thesis:** H&R Block's revenue is overwhelmingly concentrated in tax season (Jan–Apr). This creates a structural seasonality in realized volatility that is distinct from broad equity seasonality.

### HRB Fiscal Calendar (FY ends June 30)

| Fiscal Quarter | Calendar Months | Season |
|---|---|---|
| FQ1 | Jul – Sep | Dead zone |
| FQ2 | Oct – Dec | Dead zone |
| FQ3 | Jan – Mar | Peak tax season |
| FQ4 | Apr – Jun | Tax extensions / wind-down |

Section 3 of this notebook establishes the fundamental revenue seasonality from actual 10-Q/10-K filings before any vol analysis begins. The vol patterns in later sections are interpreted through that lens.

This notebook asks:

1. **Does HRB realized vol exhibit calendar-month seasonality** beyond what SPY/IWM show?
2. **Is vol compressed in the dead zone** (Jul–Dec) and elevated during tax season?
3. **Do earnings reactions differ by fiscal quarter?** (A miss in FQ3 matters more than FQ1)
4. **Is HRB vol ratio (HRB / IWM) itself seasonal?** If so, this is tradeable.

### Controls

- **SPY** — broad market seasonality baseline
- **IWM** — small-cap peer group (HRB is ~$8B market cap)

---


## 1. Configuration


In [ ]:
# ======================================================================
#  CONFIGURATION
# ======================================================================

TICKERS = ['HRB', 'SPY', 'IWM']

START_DATE = '2016-01-01'
END_DATE   = '2025-12-31'

# Rolling RV windows
RV_WINDOW  = 20   # 1-month realized vol
RV_WINDOW2 = 60   # 3-month realized vol
ANN_FACTOR = 252

# HRB fiscal year ends June 30
# Map calendar month -> HRB fiscal quarter
MONTH_TO_FQ = {
    7: 'FQ1', 8: 'FQ1', 9: 'FQ1',
    10: 'FQ2', 11: 'FQ2', 12: 'FQ2',
    1: 'FQ3', 2: 'FQ3', 3: 'FQ3',
    4: 'FQ4', 5: 'FQ4', 6: 'FQ4',
}

FQ_LABELS = {
    'FQ1': 'FQ1 (Jul\u2013Sep)\nDead Zone',
    'FQ2': 'FQ2 (Oct\u2013Dec)\nDead Zone',
    'FQ3': 'FQ3 (Jan\u2013Mar)\nPeak Tax',
    'FQ4': 'FQ4 (Apr\u2013Jun)\nWind-Down',
}

MONTH_NAMES = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

print(f"Tickers: {TICKERS}")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"RV windows: {RV_WINDOW}d, {RV_WINDOW2}d")


## 2. Setup


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

COLORS = {
    'hrb': '#1B5E20',
    'spy': '#00C4E7',
    'iwm': '#F6852B',
    'accent': '#182B40',
    'neutral': '#5A5A5A',
    'green': '#6DF2A1',
    'gold': '#F7C940',
    'pink': '#EC3586',
    'tax_season': '#4CAF50',
    'dead_zone': '#BDBDBD',
}

FQ_COLORS = {
    'FQ1': '#BDBDBD',
    'FQ2': '#9E9E9E',
    'FQ3': '#66BB6A',
    'FQ4': '#1B5E20',
}


## 3. Fundamental Revenue Seasonality — The Case from 10-Q/10-K Filings

Before analyzing any price or vol data, we establish the fundamental driver: HRB's quarterly revenue is absurdly seasonal.

The data below is sourced from H&R Block's SEC filings (10-Q and 10-K) and earnings press releases. Q4 revenue for each fiscal year is derived as: Annual Total − (Q1 + Q2 + Q3).

**Sources:**
- [H&R Block Quarterly Results](https://investors.hrblock.com/financial-information/quarterly-results)
- [FY2025 Q3 Results](https://investors.hrblock.com/news-releases/news-release-details/hr-block-reports-fiscal-2025-third-quarter-results)
- [FY2024 Q3 Results](https://investors.hrblock.com/news-releases/news-release-details/hr-block-reports-fiscal-2024-third-quarter-results-expects-be)
- [FY2023 Q3 Results](https://investors.hrblock.com/news-releases/news-release-details/hr-block-reports-fiscal-2023-third-quarter-results-provides)
- [FY2022 Q4 / Full Year Results](https://www.hrblock.com/tax-center/newsroom/around-block/fiscal-year-2022-fourth-quarter-results/)
- [FY2024 Q4 / Full Year Results](https://www.hrblock.com/tax-center/newsroom/company-news/fiscal-year-2024-fourth-quarter-results/)
- [FY2025 Q4 / Full Year Results](https://www.hrblock.com/tax-center/newsroom/company-news/fiscal-year-2025-fourth-quarter-results/)
- [FY2026 Q1 Results](https://investors.hrblock.com/news-releases/news-release-details/hr-block-reports-fiscal-2026-first-quarter-results-and-reaffirms)


In [ ]:
# ======================================================================
#  HRB Quarterly Revenue from SEC Filings (10-Q / 10-K)
#
#  Sources: H&R Block earnings press releases and SEC filings
#  Q1 = Jul-Sep, Q2 = Oct-Dec, Q3 = Jan-Mar, Q4 = Apr-Jun
#  Q4 = Annual total - (Q1 + Q2 + Q3) where direct Q4 figure unavailable
#
#  All figures in $M
# ======================================================================

hrb_rev = pd.DataFrame([
    # FY2022 (ended Jun 30, 2022) — Annual: $3,463M
    # Q1 from FY2022 Q1 press release; Q3 from FY2022 Q3 press release
    {'fy': 2022, 'fq': 'FQ1', 'quarter_end': '2021-09-30', 'revenue_m': 193},
    {'fy': 2022, 'fq': 'FQ2', 'quarter_end': '2021-12-31', 'revenue_m': 179},
    {'fy': 2022, 'fq': 'FQ3', 'quarter_end': '2022-03-31', 'revenue_m': 2103},
    {'fy': 2022, 'fq': 'FQ4', 'quarter_end': '2022-06-30', 'revenue_m': 3463 - 193 - 179 - 2103},

    # FY2023 (ended Jun 30, 2023) — Annual: $3,472M
    # Q3 from FY2023 Q3 press release: $2.1B revenue
    {'fy': 2023, 'fq': 'FQ1', 'quarter_end': '2022-09-30', 'revenue_m': 187},
    {'fy': 2023, 'fq': 'FQ2', 'quarter_end': '2022-12-31', 'revenue_m': 175},
    {'fy': 2023, 'fq': 'FQ3', 'quarter_end': '2023-03-31', 'revenue_m': 2100},
    {'fy': 2023, 'fq': 'FQ4', 'quarter_end': '2023-06-30', 'revenue_m': 3472 - 187 - 175 - 2100},

    # FY2024 (ended Jun 30, 2024) — Annual: $3,610M
    # Q1: $183.8M (press release); Q3: $2,180M (press release, beat est by 2%)
    {'fy': 2024, 'fq': 'FQ1', 'quarter_end': '2023-09-30', 'revenue_m': 184},
    {'fy': 2024, 'fq': 'FQ2', 'quarter_end': '2023-12-31', 'revenue_m': 179},
    {'fy': 2024, 'fq': 'FQ3', 'quarter_end': '2024-03-31', 'revenue_m': 2180},
    {'fy': 2024, 'fq': 'FQ4', 'quarter_end': '2024-06-30', 'revenue_m': 3610 - 184 - 179 - 2180},

    # FY2025 (ended Jun 30, 2025) — Annual: $3,761M
    # Q1: $193.8M (press release); Q3: $2,300M (press release)
    {'fy': 2025, 'fq': 'FQ1', 'quarter_end': '2024-09-30', 'revenue_m': 194},
    {'fy': 2025, 'fq': 'FQ2', 'quarter_end': '2024-12-31', 'revenue_m': 180},
    {'fy': 2025, 'fq': 'FQ3', 'quarter_end': '2025-03-31', 'revenue_m': 2300},
    {'fy': 2025, 'fq': 'FQ4', 'quarter_end': '2025-06-30', 'revenue_m': 3761 - 194 - 180 - 2300},

    # FY2026 partial — Q1 only
    {'fy': 2026, 'fq': 'FQ1', 'quarter_end': '2025-09-30', 'revenue_m': 204},
])

hrb_rev['quarter_end'] = pd.to_datetime(hrb_rev['quarter_end'])

# Compute revenue share within each fiscal year
for fy in hrb_rev['fy'].unique():
    mask = hrb_rev['fy'] == fy
    total = hrb_rev.loc[mask, 'revenue_m'].sum()
    if hrb_rev.loc[mask].shape[0] == 4:  # only compute share for full years
        hrb_rev.loc[mask, 'pct_of_annual'] = (hrb_rev.loc[mask, 'revenue_m'] / total * 100).round(1)

print("HRB Quarterly Revenue from SEC Filings ($M):\n")
display_cols = ['fy', 'fq', 'quarter_end', 'revenue_m', 'pct_of_annual']
print(hrb_rev[display_cols].to_string(index=False))

# Average revenue share by fiscal quarter
avg_share = hrb_rev.dropna(subset=['pct_of_annual']).groupby('fq')['pct_of_annual'].mean()
print(f"\nAverage Revenue Share by Fiscal Quarter (FY2022–FY2025):")
for fq in ['FQ1', 'FQ2', 'FQ3', 'FQ4']:
    bar = '█' * int(avg_share[fq] / 2)
    print(f"  {fq}: {avg_share[fq]:5.1f}%  {bar}")


In [ ]:
# Visualize the fundamental seasonality
full_years = hrb_rev[hrb_rev['fy'].isin([2022, 2023, 2024, 2025])].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# -- Panel 1: Stacked bar — revenue by FQ for each fiscal year
ax = axes[0]
fq_order = ['FQ1', 'FQ2', 'FQ3', 'FQ4']
fq_colors_plot = ['#BDBDBD', '#9E9E9E', '#1B5E20', '#66BB6A']
x = np.arange(4)  # 4 fiscal years
width = 0.6

for fy_idx, fy in enumerate([2022, 2023, 2024, 2025]):
    fy_data = full_years[full_years['fy'] == fy].set_index('fq')
    bottom = 0
    for fq_idx, fq in enumerate(fq_order):
        val = fy_data.loc[fq, 'revenue_m']
        ax.bar(fy_idx, val, width, bottom=bottom, color=fq_colors_plot[fq_idx],
               label=fq if fy_idx == 0 else '', alpha=0.85)
        if val > 300:  # label large segments
            ax.text(fy_idx, bottom + val/2, f'${val:,.0f}M',
                    ha='center', va='center', fontsize=8, fontweight='bold',
                    color='white' if fq in ['FQ3', 'FQ4'] else 'black')
        bottom += val

ax.set_xticks(x)
ax.set_xticklabels([f'FY{fy}' for fy in [2022, 2023, 2024, 2025]])
ax.set_ylabel('Revenue ($M)')
ax.set_title('HRB Revenue by Fiscal Quarter\n(from 10-Q/10-K filings)')
ax.legend(loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'${x/1000:.1f}B'))

# -- Panel 2: Average revenue share pie
ax = axes[1]
avg_shares = [avg_share[fq] for fq in fq_order]
explode = (0, 0, 0.05, 0)  # slightly explode FQ3
wedges, texts, autotexts = ax.pie(
    avg_shares, labels=None, autopct='%1.1f%%',
    colors=fq_colors_plot, explode=explode, startangle=90,
    textprops={'fontsize': 11, 'fontweight': 'bold'})
ax.legend([f'{fq} ({FQ_LABELS[fq].split(chr(10))[0].split("(")[1]}' for fq in fq_order],
          loc='lower right', fontsize=9)
ax.set_title('Average Revenue Share by Fiscal Quarter\n(FY2022–FY2025)')

plt.tight_layout()
plt.show()

print("\nKey takeaway: FQ3 (Jan–Mar) alone accounts for ~60% of annual revenue.")
print("FQ1 + FQ2 combined account for ~10%. This is the fundamental basis")
print("for expecting vol seasonality in HRB stock.")


## 4. Fetch Price Data


In [ ]:
print(f"Fetching {TICKERS} from Yahoo Finance...")
raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=True)

if isinstance(raw.columns, pd.MultiIndex):
    prices = raw['Close']
else:
    prices = raw[['Close']]
    prices.columns = TICKERS

prices = prices.dropna(how='all')
print(f"\nFetched {prices.shape[0]} trading days")
print(f"  {prices.index[0].date()} to {prices.index[-1].date()}")
print(f"\nMissing values per ticker:")
print(prices.isnull().sum().to_string())


## 5. Compute Returns & Realized Volatility


In [ ]:
# Log returns
log_ret = np.log(prices / prices.shift(1))

# Rolling realized vol (annualized %)
rv20 = log_ret.rolling(RV_WINDOW).apply(
    lambda x: np.sqrt(np.sum(x**2) / len(x)) * np.sqrt(ANN_FACTOR) * 100
)
rv20.columns = [f'{t}_rv20' for t in rv20.columns]

rv60 = log_ret.rolling(RV_WINDOW2).apply(
    lambda x: np.sqrt(np.sum(x**2) / len(x)) * np.sqrt(ANN_FACTOR) * 100
)
rv60.columns = [f'{t}_rv60' for t in rv60.columns]

# Combine into one dataframe
df = pd.concat([log_ret, rv20, rv60], axis=1)
df.columns = (
    [f'{t}_ret' for t in TICKERS] +
    [f'{t}_rv20' for t in TICKERS] +
    [f'{t}_rv60' for t in TICKERS]
)

# Calendar features
df['month'] = df.index.month
df['year'] = df.index.year
df['month_name'] = df.index.strftime('%b')
df['fq'] = df['month'].map(MONTH_TO_FQ)

# Vol ratios
df['HRB_vs_IWM_rv20'] = df['HRB_rv20'] / df['IWM_rv20']
df['HRB_vs_SPY_rv20'] = df['HRB_rv20'] / df['SPY_rv20']
df['HRB_vs_IWM_rv60'] = df['HRB_rv60'] / df['IWM_rv60']

print(f"Panel: {len(df)} rows")
print(f"\nRV20 ranges (ann %):")
for t in TICKERS:
    col = f'{t}_rv20'
    print(f"  {t}: [{df[col].min():.1f}, {df[col].max():.1f}]  median = {df[col].median():.1f}")


## 6. HRB Time Series Overview


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=True)

# -- Price
ax = axes[0]
ax.plot(prices.index, prices['HRB'], color=COLORS['hrb'], lw=1)
ax.set_ylabel('Price ($)')
ax.set_title('HRB: Price')

# Shade tax seasons (Jan-Apr each year — FQ3 + early FQ4)
for yr in range(prices.index[0].year, prices.index[-1].year + 1):
    ax.axvspan(pd.Timestamp(yr, 1, 1), pd.Timestamp(yr, 4, 30),
               alpha=0.08, color=COLORS['tax_season'], zorder=0)

# -- RV20 comparison
ax = axes[1]
ax.plot(df.index, df['HRB_rv20'], color=COLORS['hrb'], lw=0.8, alpha=0.9, label='HRB')
ax.plot(df.index, df['IWM_rv20'], color=COLORS['iwm'], lw=0.8, alpha=0.6, label='IWM')
ax.plot(df.index, df['SPY_rv20'], color=COLORS['spy'], lw=0.8, alpha=0.6, label='SPY')
ax.set_ylabel('RV20 (ann %)')
ax.set_title('20-Day Realized Volatility: HRB vs Controls')
ax.legend()

for yr in range(prices.index[0].year, prices.index[-1].year + 1):
    ax.axvspan(pd.Timestamp(yr, 1, 1), pd.Timestamp(yr, 4, 30),
               alpha=0.08, color=COLORS['tax_season'], zorder=0)

# -- Vol ratio
ax = axes[2]
ax.plot(df.index, df['HRB_vs_IWM_rv20'], color=COLORS['accent'], lw=0.8, alpha=0.8)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.axhline(df['HRB_vs_IWM_rv20'].median(), color=COLORS['iwm'], ls='-', lw=1, alpha=0.5,
           label=f'Median = {df["HRB_vs_IWM_rv20"].median():.2f}')
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title('HRB Vol Ratio vs IWM')
ax.legend()

for yr in range(prices.index[0].year, prices.index[-1].year + 1):
    ax.axvspan(pd.Timestamp(yr, 1, 1), pd.Timestamp(yr, 4, 30),
               alpha=0.08, color=COLORS['tax_season'], zorder=0)

plt.tight_layout()
plt.show()
print("Green shading = tax season (Jan–Apr)")


## 7. Monthly Seasonality — Realized Vol by Calendar Month

The core question: does HRB's realized vol have a month-of-year pattern that differs from broad equity?


In [ ]:
monthly_rv = df.dropna(subset=['HRB_rv20']).groupby('month').agg(
    HRB_rv20_mean=('HRB_rv20', 'mean'),
    HRB_rv20_med=('HRB_rv20', 'median'),
    IWM_rv20_mean=('IWM_rv20', 'mean'),
    IWM_rv20_med=('IWM_rv20', 'median'),
    SPY_rv20_mean=('SPY_rv20', 'mean'),
    SPY_rv20_med=('SPY_rv20', 'median'),
    n=('HRB_rv20', 'count'),
).round(2)

print("Monthly Median RV20 (ann %):\n")
print(monthly_rv.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

months = range(1, 13)
x = np.arange(12)
w = 0.25

# -- Panel 1: Median RV20 by month
ax = axes[0]
hrb_vals = [monthly_rv.loc[m, 'HRB_rv20_med'] for m in months]
iwm_vals = [monthly_rv.loc[m, 'IWM_rv20_med'] for m in months]
spy_vals = [monthly_rv.loc[m, 'SPY_rv20_med'] for m in months]

ax.bar(x - w, hrb_vals, w, label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, iwm_vals, w, label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, spy_vals, w, label='SPY', color=COLORS['spy'], alpha=0.7)

# Shade tax season months
for m_idx in [0, 1, 2, 3]:  # Jan-Apr
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Median RV20 (ann %)')
ax.set_title('Median 20-Day Realized Vol by Calendar Month')
ax.legend()

# -- Panel 2: HRB excess vol over IWM by month
ax = axes[1]
excess = [hrb_vals[i] - iwm_vals[i] for i in range(12)]
bar_colors = [COLORS['hrb'] if e > 0 else COLORS['dead_zone'] for e in excess]
ax.bar(x, excess, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(0, color='black', lw=1)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB Median RV20 − IWM Median RV20 (pp)')
ax.set_title('HRB Excess Volatility Over IWM by Month')

plt.tight_layout()
plt.show()
print("Green shading = tax season months (Jan–Apr)")


## 8. Monthly Seasonality — Returns

Does HRB show return seasonality in addition to vol seasonality? Tax season earnings drive the stock — do returns cluster?


In [ ]:
# Monthly simple returns (sum of daily log returns per calendar month)
monthly_ret = df.groupby([df.index.year, df.index.month]).agg(
    HRB_ret=('HRB_ret', 'sum'),
    IWM_ret=('IWM_ret', 'sum'),
    SPY_ret=('SPY_ret', 'sum'),
)
monthly_ret.index.names = ['year', 'month']
monthly_ret = monthly_ret.reset_index()
# Convert log returns to simple returns for display
for col in ['HRB_ret', 'IWM_ret', 'SPY_ret']:
    monthly_ret[col] = (np.exp(monthly_ret[col]) - 1) * 100  # percent

monthly_ret_season = monthly_ret.groupby('month').agg(
    HRB_mean=('HRB_ret', 'mean'),
    HRB_med=('HRB_ret', 'median'),
    HRB_std=('HRB_ret', 'std'),
    HRB_pct_pos=('HRB_ret', lambda x: (x > 0).mean() * 100),
    IWM_mean=('IWM_ret', 'mean'),
    IWM_med=('IWM_ret', 'median'),
    SPY_mean=('SPY_ret', 'mean'),
    SPY_med=('SPY_ret', 'median'),
    n=('HRB_ret', 'count'),
).round(2)

print("Monthly Return Seasonality (%):\n")
print(monthly_ret_season.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

x = np.arange(12)
w = 0.25

# -- Panel 1: Mean monthly returns
ax = axes[0]
hrb_r = [monthly_ret_season.loc[m, 'HRB_mean'] for m in range(1, 13)]
iwm_r = [monthly_ret_season.loc[m, 'IWM_mean'] for m in range(1, 13)]
spy_r = [monthly_ret_season.loc[m, 'SPY_mean'] for m in range(1, 13)]

ax.bar(x - w, hrb_r, w, label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, iwm_r, w, label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, spy_r, w, label='SPY', color=COLORS['spy'], alpha=0.7)
ax.axhline(0, color='black', lw=1)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Mean Monthly Return (%)')
ax.set_title('Mean Monthly Return by Calendar Month')
ax.legend()

# -- Panel 2: HRB % positive months
ax = axes[1]
pct_pos = [monthly_ret_season.loc[m, 'HRB_pct_pos'] for m in range(1, 13)]
bar_colors = [COLORS['hrb'] if p > 50 else COLORS['pink'] for p in pct_pos]
ax.bar(x, pct_pos, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5, label='50%')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('% of Months Positive')
ax.set_title('HRB: Win Rate by Calendar Month')
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.show()


## 9. Fiscal Quarter Analysis

Aggregating by HRB fiscal quarter to see the dead-zone vs tax-season contrast.


In [ ]:
fq_stats = df.dropna(subset=['HRB_rv20']).groupby('fq').agg(
    n=('HRB_rv20', 'count'),
    HRB_rv20_med=('HRB_rv20', 'median'),
    HRB_rv20_mean=('HRB_rv20', 'mean'),
    HRB_rv20_p25=('HRB_rv20', lambda x: np.percentile(x, 25)),
    HRB_rv20_p75=('HRB_rv20', lambda x: np.percentile(x, 75)),
    IWM_rv20_med=('IWM_rv20', 'median'),
    SPY_rv20_med=('SPY_rv20', 'median'),
    HRB_ret_mean=('HRB_ret', lambda x: x.mean() * ANN_FACTOR * 100),
    vol_ratio_med=('HRB_vs_IWM_rv20', 'median'),
).reindex(['FQ1', 'FQ2', 'FQ3', 'FQ4']).round(2)

print("HRB Fiscal Quarter Summary:\n")
print(fq_stats.to_string())


In [ ]:
fq_order = ['FQ1', 'FQ2', 'FQ3', 'FQ4']

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# -- Panel 1: RV20 distributions by FQ (boxplot style)
ax = axes[0]
fq_data = [df.dropna(subset=['HRB_rv20']).loc[df.fq == fq, 'HRB_rv20'].values for fq in fq_order]
bp = ax.boxplot(fq_data, positions=range(4), widths=0.5, patch_artist=True,
                showfliers=False, medianprops={'color': 'black', 'lw': 2})
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(FQ_COLORS[fq_order[i]])
    patch.set_alpha(0.7)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('HRB RV20 (ann %)')
ax.set_title('HRB RV20 Distribution by Fiscal Quarter')

# -- Panel 2: HRB vs IWM vol ratio by FQ
ax = axes[1]
ratio_data = [df.dropna(subset=['HRB_vs_IWM_rv20']).loc[df.fq == fq, 'HRB_vs_IWM_rv20'].values for fq in fq_order]
bp2 = ax.boxplot(ratio_data, positions=range(4), widths=0.5, patch_artist=True,
                 showfliers=False, medianprops={'color': 'black', 'lw': 2})
for i, patch in enumerate(bp2['boxes']):
    patch.set_facecolor(FQ_COLORS[fq_order[i]])
    patch.set_alpha(0.7)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title('HRB Relative Vol (vs IWM) by Fiscal Quarter')

# -- Panel 3: Median comparison HRB vs controls
ax = axes[2]
x = np.arange(4)
w = 0.25
ax.bar(x - w, [fq_stats.loc[fq, 'HRB_rv20_med'] for fq in fq_order], w,
       label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, [fq_stats.loc[fq, 'IWM_rv20_med'] for fq in fq_order], w,
       label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, [fq_stats.loc[fq, 'SPY_rv20_med'] for fq in fq_order], w,
       label='SPY', color=COLORS['spy'], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('Median RV20 (ann %)')
ax.set_title('Median RV20 by Fiscal Quarter: HRB vs Controls')
ax.legend()

plt.tight_layout()
plt.show()


## 10. Vol Ratio Seasonality — Is HRB/IWM Itself Seasonal?

If the vol ratio (HRB RV20 / IWM RV20) is systematically higher in certain months, that's a tradeable signal: sell HRB vol relative to index vol in the dead zone, buy it into tax season.


In [ ]:
vol_ratio_monthly = df.dropna(subset=['HRB_vs_IWM_rv20']).groupby('month').agg(
    ratio_mean=('HRB_vs_IWM_rv20', 'mean'),
    ratio_med=('HRB_vs_IWM_rv20', 'median'),
    ratio_p25=('HRB_vs_IWM_rv20', lambda x: np.percentile(x, 25)),
    ratio_p75=('HRB_vs_IWM_rv20', lambda x: np.percentile(x, 75)),
    n=('HRB_vs_IWM_rv20', 'count'),
).round(3)

print("HRB/IWM Vol Ratio by Calendar Month:\n")
print(vol_ratio_monthly.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(12)
meds = [vol_ratio_monthly.loc[m, 'ratio_med'] for m in range(1, 13)]
p25 = [vol_ratio_monthly.loc[m, 'ratio_p25'] for m in range(1, 13)]
p75 = [vol_ratio_monthly.loc[m, 'ratio_p75'] for m in range(1, 13)]

# IQR bands
ax.fill_between(x, p25, p75, alpha=0.2, color=COLORS['hrb'], label='p25–p75')
ax.plot(x, meds, 'o-', color=COLORS['hrb'], lw=2, markersize=8, label='Median', zorder=5)

ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5, label='Ratio = 1.0')

# Shade tax season
for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title('HRB/IWM Vol Ratio Seasonality by Calendar Month\n(Green shading = tax season)')
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Box plot: HRB/IWM vol ratio by calendar month
fig, ax = plt.subplots(figsize=(14, 7))

ratio_by_month = [df.dropna(subset=['HRB_vs_IWM_rv20']).loc[df.month == m, 'HRB_vs_IWM_rv20'].values
                  for m in range(1, 13)]

bp = ax.boxplot(ratio_by_month, positions=range(12), widths=0.5, patch_artist=True,
                medianprops={'color': COLORS['accent'], 'lw': 2})

for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor('#B0D4F1')
    patch.set_alpha(0.7)
    patch.set_edgecolor('#7EB3DA')

# Shade tax season
for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB RV20 / IWM RV20')
ax.set_title(f'HRB/IWM vol ratio by calendar month ({START_DATE[:4]}–{END_DATE[:4]})')

plt.tight_layout()
plt.show()

## 11. Statistical Significance — Log Vol Ratio by Month

Working in log-space (log(HRB RV20 / IWM RV20)) makes the ratio symmetric and approximately normal.

**The overlap problem:** RV20 uses a 20-day rolling window, so consecutive daily observations share 19 of 20 days. 191 daily readings in February aren't 191 independent data points — they're roughly ~10 (one per year). A naive t-test on daily data dramatically overstates confidence.

**Solution:** We collapse to **one observation per year×month** (the median log-ratio for that month in that year). With ~10 years of data, each month gets ~10 truly independent observations. The t-test and p-values are now honest.

- **Cohen's d** (effect size): how many standard deviations the month's mean sits from the grand mean. Measures *magnitude*. |d| > 0.2 is "small," > 0.5 is "medium," > 0.8 is "large."
- **t-statistic**: with proper n (~10), this is no longer inflated. A |t| > 2 on 9 df is genuinely meaningful.
- **Percentile**: where the month's median log-ratio sits in the distribution of all year×month medians.

In [ ]:
from scipy import stats as sp_stats

# Log vol ratio
df['log_ratio'] = np.log(df['HRB_vs_IWM_rv20'])
valid_lr = df.dropna(subset=['log_ratio']).copy()

# ======================================================================
#  Collapse to year×month medians — one independent observation per
#  year×month cell.  With a 20-day rolling window, daily readings share
#  19/20 days; the median of each year×month block is a single honest
#  data point.
# ======================================================================
ym_medians = (
    valid_lr
    .groupby(['year', 'month'])['log_ratio']
    .median()
    .reset_index(name='log_ratio_med')
)

# Grand statistics (pooled across all year×month medians)
grand_mean = ym_medians['log_ratio_med'].mean()
grand_std  = ym_medians['log_ratio_med'].std()
grand_n    = len(ym_medians)

# Per-month statistics on the independent year×month medians
log_ratio_stats = pd.DataFrame(index=range(1, 13))

for m in range(1, 13):
    month_vals = ym_medians.loc[ym_medians.month == m, 'log_ratio_med']
    n_m   = len(month_vals)
    mean_m = month_vals.mean()
    std_m  = month_vals.std()
    med_m  = month_vals.median()

    # Cohen's d: (month mean − grand mean) / grand std
    cohens_d = (mean_m - grand_mean) / grand_std

    # t-test: is this month's mean different from the grand mean?
    # Now n_m ≈ 10 (one per year), so degrees of freedom are honest
    se_m   = std_m / np.sqrt(n_m)
    t_stat = (mean_m - grand_mean) / se_m
    # Use t-distribution with n_m-1 df (not normal approx)
    p_value = 2 * sp_stats.t.sf(abs(t_stat), df=n_m - 1)

    # Percentile: where does this month's median sit among all year×month medians?
    percentile = sp_stats.percentileofscore(ym_medians['log_ratio_med'].values, med_m)

    log_ratio_stats.loc[m, 'n'] = n_m
    log_ratio_stats.loc[m, 'mean_ratio'] = np.exp(mean_m)
    log_ratio_stats.loc[m, 'med_ratio']  = np.exp(med_m)
    log_ratio_stats.loc[m, 'mean_log']   = mean_m
    log_ratio_stats.loc[m, 'std_log']    = std_m
    log_ratio_stats.loc[m, 'cohens_d']   = cohens_d
    log_ratio_stats.loc[m, 't_stat']     = t_stat
    log_ratio_stats.loc[m, 'p_value']    = p_value
    log_ratio_stats.loc[m, 'percentile'] = percentile

log_ratio_stats['n'] = log_ratio_stats['n'].astype(int)

def d_label(d):
    ad = abs(d)
    if ad >= 0.8: return 'large'
    if ad >= 0.5: return 'medium'
    if ad >= 0.2: return 'small'
    return '~0'

log_ratio_stats['effect'] = log_ratio_stats['cohens_d'].apply(d_label)
log_ratio_stats['sig'] = log_ratio_stats['p_value'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
)

print(f"Grand mean log(HRB/IWM) [year×month medians]: {grand_mean:.4f}  (ratio = {np.exp(grand_mean):.3f})")
print(f"Grand std:               {grand_std:.4f}")
print(f"N year×month cells:      {grand_n}  (~{grand_n//12} years × 12 months)\n")

display_df = log_ratio_stats.copy()
display_df.index = MONTH_NAMES
print(display_df[['n', 'med_ratio', 'mean_log', 'cohens_d', 'effect', 't_stat', 'p_value', 'percentile', 'sig']].round(4).to_string())

print("\nCohen's d: |d|>0.2 small, >0.5 medium, >0.8 large")
print("Significance (t-distribution, n-1 df): *** p<0.01, ** p<0.05, * p<0.1")
print(f"\nEach month has n ≈ {log_ratio_stats['n'].median():.0f} independent observations (one per year).")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))

x = np.arange(12)

# -- Panel 1: Cohen's d (effect size) by month
ax = axes[0]
d_vals = [log_ratio_stats.loc[m, 'cohens_d'] for m in range(1, 13)]
bar_colors = []
for d in d_vals:
    if abs(d) >= 0.5:
        bar_colors.append(COLORS['hrb'] if d > 0 else COLORS['pink'])
    elif abs(d) >= 0.2:
        bar_colors.append('#A5D6A7' if d > 0 else '#EF9A9A')
    else:
        bar_colors.append(COLORS['dead_zone'])

ax.bar(x, d_vals, 0.6, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', lw=1)
ax.axhline(0.5, color=COLORS['neutral'], ls='--', lw=1, alpha=0.4, label='|d| = 0.5 (medium)')
ax.axhline(-0.5, color=COLORS['neutral'], ls='--', lw=1, alpha=0.4)
ax.axhline(0.2, color=COLORS['neutral'], ls=':', lw=1, alpha=0.3, label='|d| = 0.2 (small)')
ax.axhline(-0.2, color=COLORS['neutral'], ls=':', lw=1, alpha=0.3)

for i, d in enumerate(d_vals):
    ax.text(i, d + (0.03 if d >= 0 else -0.06), f'{d:.2f}', ha='center', fontsize=8, fontweight='bold')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel("Cohen's d")
ax.set_title("Effect Size: Month Mean log(HRB/IWM) vs Grand Mean\n(year×month medians, not inflated by overlapping data)")
ax.legend(fontsize=8)

# -- Panel 2: Percentile by month
ax = axes[1]
pct_vals = [log_ratio_stats.loc[m, 'percentile'] for m in range(1, 13)]
pct_colors = [COLORS['hrb'] if p > 60 else (COLORS['pink'] if p < 40 else COLORS['dead_zone']) for p in pct_vals]
ax.bar(x, pct_vals, 0.6, color=pct_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5, label='50th percentile')

for i, p in enumerate(pct_vals):
    ax.text(i, p + 1.5, f'{p:.0f}', ha='center', fontsize=9, fontweight='bold')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Percentile in Overall Distribution')
ax.set_title('Percentile: Month Median log(HRB/IWM)\nvs All Year×Month Medians')
ax.set_ylim(0, 100)
ax.legend(fontsize=8)

# -- Panel 3: Year×month median distributions by month (strip/swarm plot)
#    Each dot = one year's median log-ratio for that month (~10 dots per month)
ax = axes[2]
for m in range(1, 13):
    month_pts = ym_medians.loc[ym_medians.month == m, 'log_ratio_med'].values
    # Jitter x position slightly for visibility
    jitter = np.random.default_rng(m).uniform(-0.15, 0.15, size=len(month_pts))
    d = abs(log_ratio_stats.loc[m, 'cohens_d'])
    color = COLORS['hrb'] if d >= 0.2 else COLORS['dead_zone']
    ax.scatter(m - 1 + jitter, month_pts, color=color, alpha=0.6, s=30, zorder=3)
    # Mark the month mean
    ax.plot([m - 1 - 0.25, m - 1 + 0.25],
            [month_pts.mean(), month_pts.mean()],
            color=COLORS['accent'], lw=2, zorder=4)

ax.axhline(grand_mean, color=COLORS['pink'], ls='--', lw=1, alpha=0.7,
           label=f'Grand mean = {grand_mean:.3f}')
ax.axhline(0, color=COLORS['neutral'], ls=':', lw=1, alpha=0.3, label='log(ratio) = 0 (ratio = 1)')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('log(HRB RV20 / IWM RV20)')
ax.set_title('Year×Month Medians of log(HRB/IWM)\n(each dot = one year, horizontal bar = month mean)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary using effect size
medium_high = [MONTH_NAMES[m-1] for m in range(1, 13) if log_ratio_stats.loc[m, 'cohens_d'] >= 0.2]
medium_low = [MONTH_NAMES[m-1] for m in range(1, 13) if log_ratio_stats.loc[m, 'cohens_d'] <= -0.2]
print(f"Months with ELEVATED ratio (d >= 0.2):  {', '.join(medium_high) if medium_high else 'None'}")
print(f"Months with DEPRESSED ratio (d <= -0.2): {', '.join(medium_low) if medium_low else 'None'}")
n_per = log_ratio_stats['n'].median()
print(f"\nAll stats computed on ~{n_per:.0f} independent year×month medians per month.")
print(f"t-tests use t-distribution with n-1 ≈ {n_per-1:.0f} df — no longer inflated by overlapping data.")

## 12. Earnings Event Analysis

HRB reports earnings 4 times per year. The market's reaction should differ by fiscal quarter: a miss on the FQ3 (peak tax) print carries far more information than an FQ1 (dead zone) miss.

We identify likely earnings days from large single-day moves (|daily return| > 2 standard deviations) and analyze them by fiscal quarter.

In [ ]:
# Identify likely earnings days: large single-day HRB moves
hrb_ret = df['HRB_ret'].dropna()
ret_std = hrb_ret.std()
ret_mean = hrb_ret.mean()
threshold = 2.0 * ret_std

big_days = df[abs(df['HRB_ret']) > threshold].copy()
big_days['abs_ret'] = abs(big_days['HRB_ret']) * 100
big_days['direction'] = np.where(big_days['HRB_ret'] > 0, 'Up', 'Down')

print(f"Threshold: |return| > {threshold*100:.2f}% (2 sigma)")
print(f"Big move days: {len(big_days)} out of {len(hrb_ret)} trading days ({len(big_days)/len(hrb_ret)*100:.1f}%)")
print(f"\nBy fiscal quarter:")
for fq in ['FQ1', 'FQ2', 'FQ3', 'FQ4']:
    sub = big_days[big_days.fq == fq]
    print(f"  {fq}: {len(sub)} big days, "
          f"median |move| = {sub.abs_ret.median():.1f}%, "
          f"up/down = {(sub.direction == 'Up').sum()}/{(sub.direction == 'Down').sum()}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# -- Panel 1: Distribution of big-day moves by FQ
ax = axes[0]
for fq in ['FQ1', 'FQ2', 'FQ3', 'FQ4']:
    sub = big_days[big_days.fq == fq]
    ax.hist(sub['HRB_ret'] * 100, bins=30, range=(-15, 15), alpha=0.4,
            color=FQ_COLORS[fq], label=f'{fq} (n={len(sub)})', density=True)
ax.axvline(0, color='black', lw=1)
ax.set_xlabel('HRB Daily Return (%)')
ax.set_ylabel('Density')
ax.set_title('Distribution of Big Move Days (>2σ) by Fiscal Quarter')
ax.legend()

# -- Panel 2: Median absolute return on big days by FQ
ax = axes[1]
fq_big = big_days.groupby('fq').agg(
    count=('abs_ret', 'count'),
    med_abs_ret=('abs_ret', 'median'),
    mean_abs_ret=('abs_ret', 'mean'),
).reindex(['FQ1', 'FQ2', 'FQ3', 'FQ4'])

ax.bar(range(4), fq_big['med_abs_ret'], 0.6,
       color=[FQ_COLORS[fq] for fq in fq_order], alpha=0.8)
for i, fq in enumerate(fq_order):
    ax.text(i, fq_big.loc[fq, 'med_abs_ret'] + 0.1,
            f'n={int(fq_big.loc[fq, "count"])}', ha='center', fontsize=10)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('Median |Return| on Big Days (%)')
ax.set_title('Median Magnitude of Big Moves by Fiscal Quarter')

plt.tight_layout()
plt.show()


## 13. Year-Over-Year Heatmap — Monthly RV20

Each cell shows HRB's median RV20 for that year x month. This reveals whether the seasonal pattern is stable across years or driven by a few outlier years.

In [ ]:
ym_rv = df.dropna(subset=['HRB_rv20']).groupby(['year', 'month'])['HRB_rv20'].median().unstack()

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_rv) * 0.5)))

vals = ym_rv.values
im = ax.imshow(vals, cmap='YlOrRd', aspect='auto',
               vmin=np.nanpercentile(vals, 5), vmax=np.nanpercentile(vals, 95))

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if v > np.nanpercentile(vals, 70) else 'black'
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_rv)))
ax.set_yticklabels(ym_rv.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title('HRB Median RV20 (ann %) by Year × Month')
plt.colorbar(im, shrink=0.8, label='RV20 (ann %)')

plt.tight_layout()
plt.show()


## 14. Year-Over-Year Heatmap — Vol Ratio (HRB / IWM)

Same heatmap but for the vol ratio. This controls for market-wide vol regimes and isolates the HRB-specific seasonal component.

In [ ]:
ym_ratio = df.dropna(subset=['HRB_vs_IWM_rv20']).groupby(['year', 'month'])['HRB_vs_IWM_rv20'].median().unstack()

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_ratio) * 0.5)))

vals = ym_ratio.values
vmax = max(abs(np.nanpercentile(vals, 5) - 1), abs(np.nanpercentile(vals, 95) - 1))
im = ax.imshow(vals, cmap='RdYlGn_r', aspect='auto',
               vmin=1 - vmax, vmax=1 + vmax)

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if abs(v - 1) > vmax * 0.5 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=8, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_ratio)))
ax.set_yticklabels(ym_ratio.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title('HRB/IWM Vol Ratio by Year × Month\n(>1 = HRB more volatile than IWM)')
plt.colorbar(im, shrink=0.8, label='HRB RV20 / IWM RV20')

plt.tight_layout()
plt.show()


## 15. Cumulative Return by Fiscal Quarter — Tax Season Alpha?

If we only held HRB during tax season (FQ3+FQ4) vs. only during the dead zone (FQ1+FQ2), what would the return profile look like?

In [ ]:
df_valid = df.dropna(subset=['HRB_ret']).copy()

# Tax season: FQ3 + FQ4 (Jan-Jun)
# Dead zone: FQ1 + FQ2 (Jul-Dec)
df_valid['tax_season'] = df_valid['fq'].isin(['FQ3', 'FQ4'])

df_valid['hrb_tax_ret'] = np.where(df_valid['tax_season'], df_valid['HRB_ret'], 0)
df_valid['hrb_dead_ret'] = np.where(~df_valid['tax_season'], df_valid['HRB_ret'], 0)
df_valid['hrb_all_ret'] = df_valid['HRB_ret']

# Cumulative returns
df_valid['cum_tax'] = df_valid['hrb_tax_ret'].cumsum()
df_valid['cum_dead'] = df_valid['hrb_dead_ret'].cumsum()
df_valid['cum_all'] = df_valid['hrb_all_ret'].cumsum()
df_valid['cum_iwm'] = df_valid['IWM_ret'].cumsum()

fig, ax = plt.subplots(figsize=(16, 8))
ax.plot(df_valid.index, (np.exp(df_valid['cum_tax']) - 1) * 100,
        color=COLORS['hrb'], lw=1.5, label='HRB: Tax Season Only (Jan–Jun)')
ax.plot(df_valid.index, (np.exp(df_valid['cum_dead']) - 1) * 100,
        color=COLORS['dead_zone'], lw=1.5, label='HRB: Dead Zone Only (Jul–Dec)')
ax.plot(df_valid.index, (np.exp(df_valid['cum_all']) - 1) * 100,
        color=COLORS['accent'], lw=1, alpha=0.5, label='HRB: All Year')
ax.plot(df_valid.index, (np.exp(df_valid['cum_iwm']) - 1) * 100,
        color=COLORS['iwm'], lw=1, alpha=0.5, label='IWM: All Year')

ax.axhline(0, color='black', lw=0.5)
ax.set_ylabel('Cumulative Return (%)')
ax.set_title('HRB Cumulative Returns: Tax Season vs Dead Zone')
ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

# Annualized stats
years = (df_valid.index[-1] - df_valid.index[0]).days / 365.25
for label, col in [('Tax Season', 'cum_tax'), ('Dead Zone', 'cum_dead'), ('All Year', 'cum_all')]:
    total = df_valid[col].iloc[-1]
    ann_ret = (np.exp(total / years) - 1) * 100
    print(f"  HRB {label}: {(np.exp(total)-1)*100:.1f}% total, ~{ann_ret:.1f}% ann")


## 16. Summary Statistics

In [ ]:
print("=" * 70)
print("HRB SEASONALITY SUMMARY")
print("=" * 70)

valid = df.dropna(subset=['HRB_rv20', 'IWM_rv20'])

tax = valid[valid.fq.isin(['FQ3', 'FQ4'])]
dead = valid[valid.fq.isin(['FQ1', 'FQ2'])]

print(f"\nHRB Median RV20:")
print(f"  Tax Season (Jan–Jun):  {tax['HRB_rv20'].median():.1f}%")
print(f"  Dead Zone  (Jul–Dec):  {dead['HRB_rv20'].median():.1f}%")
print(f"  Ratio (tax/dead):      {tax['HRB_rv20'].median() / dead['HRB_rv20'].median():.2f}x")

print(f"\nIWM Median RV20 (control):")
print(f"  Tax Season (Jan–Jun):  {tax['IWM_rv20'].median():.1f}%")
print(f"  Dead Zone  (Jul–Dec):  {dead['IWM_rv20'].median():.1f}%")
print(f"  Ratio (tax/dead):      {tax['IWM_rv20'].median() / dead['IWM_rv20'].median():.2f}x")

print(f"\nHRB/IWM Vol Ratio:")
print(f"  Tax Season median:     {tax['HRB_vs_IWM_rv20'].median():.3f}")
print(f"  Dead Zone  median:     {dead['HRB_vs_IWM_rv20'].median():.3f}")

print(f"\nHRB Mean Daily Return (annualized):")
print(f"  Tax Season: {tax['HRB_ret'].mean() * ANN_FACTOR * 100:.1f}%")
print(f"  Dead Zone:  {dead['HRB_ret'].mean() * ANN_FACTOR * 100:.1f}%")
print(f"\n" + "=" * 70)


## 18. Calendar-Month Realized Volatility — Non-Overlapping Alternative

The rolling 20-day RV used throughout this notebook produces smooth daily time series but creates massive overlap: consecutive observations share 19 of 20 days. This inflates apparent sample sizes and makes box plots noisy with pseudo-outliers.

**Alternative approach:** compute one RV estimate per calendar month per ticker:

$$\text{RV}_{\text{month}} = \sqrt{\frac{1}{N}\sum_{i=1}^{N} r_i^2} \times \sqrt{252} \times 100$$

where $r_i$ are daily log returns within that calendar month and $N$ is the number of trading days.

**Advantages:**
- Each observation is **fully independent** — no overlap whatsoever
- ~10 observations per calendar month (one per year) — honest sample sizes
- No arbitrary window choice; the calendar month *is* the natural unit
- Statistical tests are valid without adjustment

**Tradeoff:** Fewer data points per month (just ~10 years), so patterns need to be robust to survive.

This section mirrors the vol-focused charts from Sections 7, 9, 10, 11, 13, and 14 using this cleaner measure.

In [ ]:
# ======================================================================
#  Compute calendar-month realized vol for each ticker
#  RV_month = sqrt(mean(r^2)) * sqrt(252) * 100
# ======================================================================

cmrv = (
    df[['HRB_ret', 'IWM_ret', 'SPY_ret', 'month', 'year', 'fq']]
    .dropna(subset=['HRB_ret'])
    .copy()
)

# One RV per year×month per ticker
cmrv_panel = (
    cmrv.groupby(['year', 'month'])
    .agg(
        HRB_cmrv=('HRB_ret', lambda x: np.sqrt(np.mean(x**2)) * np.sqrt(ANN_FACTOR) * 100),
        IWM_cmrv=('IWM_ret', lambda x: np.sqrt(np.mean(x**2)) * np.sqrt(ANN_FACTOR) * 100),
        SPY_cmrv=('SPY_ret', lambda x: np.sqrt(np.mean(x**2)) * np.sqrt(ANN_FACTOR) * 100),
        n_days=('HRB_ret', 'count'),
        fq=('fq', 'first'),
    )
    .reset_index()
)

# Vol ratio
cmrv_panel['HRB_vs_IWM'] = cmrv_panel['HRB_cmrv'] / cmrv_panel['IWM_cmrv']
cmrv_panel['HRB_vs_SPY'] = cmrv_panel['HRB_cmrv'] / cmrv_panel['SPY_cmrv']
cmrv_panel['log_ratio'] = np.log(cmrv_panel['HRB_vs_IWM'])

print(f"Calendar-month RV panel: {len(cmrv_panel)} year×month cells")
print(f"Trading days per month: {cmrv_panel.n_days.min()}–{cmrv_panel.n_days.max()} "
      f"(median {cmrv_panel.n_days.median():.0f})")
print(f"\nCalendar-Month RV ranges (ann %):")
for t in TICKERS:
    col = f'{t}_cmrv'
    print(f"  {t}: [{cmrv_panel[col].min():.1f}, {cmrv_panel[col].max():.1f}]  "
          f"median = {cmrv_panel[col].median():.1f}")
print(f"\nHRB/IWM ratio: [{cmrv_panel.HRB_vs_IWM.min():.2f}, {cmrv_panel.HRB_vs_IWM.max():.2f}]  "
      f"median = {cmrv_panel.HRB_vs_IWM.median():.2f}")

# Preview
print(f"\nFirst few rows:")
print(cmrv_panel.head(10).round(2).to_string(index=False))

In [ ]:
# ======================================================================
#  18a. Monthly Seasonality — Calendar-Month RV by Calendar Month
#       (mirrors Section 7)
# ======================================================================

cmrv_by_month = cmrv_panel.groupby('month').agg(
    HRB_mean=('HRB_cmrv', 'mean'),
    HRB_med=('HRB_cmrv', 'median'),
    IWM_mean=('IWM_cmrv', 'mean'),
    IWM_med=('IWM_cmrv', 'median'),
    SPY_mean=('SPY_cmrv', 'mean'),
    SPY_med=('SPY_cmrv', 'median'),
    n=('HRB_cmrv', 'count'),
).round(2)

print("Calendar-Month RV by Month (ann %):\n")
print(cmrv_by_month.to_string())

# --- Chart ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
x = np.arange(12)
w = 0.25

# Panel 1: Median calendar-month RV
ax = axes[0]
hrb_v = [cmrv_by_month.loc[m, 'HRB_med'] for m in range(1, 13)]
iwm_v = [cmrv_by_month.loc[m, 'IWM_med'] for m in range(1, 13)]
spy_v = [cmrv_by_month.loc[m, 'SPY_med'] for m in range(1, 13)]

ax.bar(x - w, hrb_v, w, label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, iwm_v, w, label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, spy_v, w, label='SPY', color=COLORS['spy'], alpha=0.7)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Median Calendar-Month RV (ann %)')
ax.set_title('Median Calendar-Month Realized Vol by Month\n(one observation per year×month — no overlap)')
ax.legend()

# Panel 2: HRB excess over IWM
ax = axes[1]
excess = [hrb_v[i] - iwm_v[i] for i in range(12)]
bar_colors = [COLORS['hrb'] if e > 0 else COLORS['dead_zone'] for e in excess]
ax.bar(x, excess, 0.6, color=bar_colors, alpha=0.8)
ax.axhline(0, color='black', lw=1)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB Median CMRV − IWM Median CMRV (pp)')
ax.set_title('HRB Excess Calendar-Month Vol Over IWM by Month')

plt.tight_layout()
plt.show()
print("Green shading = tax season months (Jan–Apr)")
print(f"n = {cmrv_by_month['n'].iloc[0]} independent observations per month")

In [ ]:
# ======================================================================
#  18b. Fiscal Quarter Analysis — Calendar-Month RV
#       (mirrors Section 9)
# ======================================================================

fq_order = ['FQ1', 'FQ2', 'FQ3', 'FQ4']

fq_cmrv_stats = cmrv_panel.groupby('fq').agg(
    n=('HRB_cmrv', 'count'),
    HRB_med=('HRB_cmrv', 'median'),
    HRB_mean=('HRB_cmrv', 'mean'),
    HRB_p25=('HRB_cmrv', lambda x: np.percentile(x, 25)),
    HRB_p75=('HRB_cmrv', lambda x: np.percentile(x, 75)),
    IWM_med=('IWM_cmrv', 'median'),
    SPY_med=('SPY_cmrv', 'median'),
    ratio_med=('HRB_vs_IWM', 'median'),
).reindex(fq_order).round(2)

print("Calendar-Month RV by Fiscal Quarter:\n")
print(fq_cmrv_stats.to_string())

# --- Chart ---
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Panel 1: HRB CMRV distributions by FQ (strip + box)
ax = axes[0]
for i, fq in enumerate(fq_order):
    vals = cmrv_panel.loc[cmrv_panel.fq == fq, 'HRB_cmrv'].values
    jitter = np.random.default_rng(i).uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(np.full_like(vals, i) + jitter, vals,
               color=FQ_COLORS[fq], alpha=0.6, s=40, zorder=3, edgecolors='white', linewidth=0.5)
    ax.plot([i - 0.2, i + 0.2], [np.median(vals)] * 2,
            color='black', lw=2.5, zorder=4)

ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('HRB Calendar-Month RV (ann %)')
ax.set_title('HRB Calendar-Month RV by Fiscal Quarter\n(each dot = one month, bar = median)')

# Panel 2: HRB vs IWM vol ratio by FQ
ax = axes[1]
for i, fq in enumerate(fq_order):
    vals = cmrv_panel.loc[cmrv_panel.fq == fq, 'HRB_vs_IWM'].values
    jitter = np.random.default_rng(i + 100).uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(np.full_like(vals, i) + jitter, vals,
               color=FQ_COLORS[fq], alpha=0.6, s=40, zorder=3, edgecolors='white', linewidth=0.5)
    ax.plot([i - 0.2, i + 0.2], [np.median(vals)] * 2,
            color='black', lw=2.5, zorder=4)

ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.set_xticks(range(4))
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('HRB CMRV / IWM CMRV')
ax.set_title('HRB Relative Vol (vs IWM) by Fiscal Quarter\n(calendar-month RV)')

# Panel 3: Median comparison HRB vs controls
ax = axes[2]
x = np.arange(4)
w = 0.25
ax.bar(x - w, [fq_cmrv_stats.loc[fq, 'HRB_med'] for fq in fq_order], w,
       label='HRB', color=COLORS['hrb'], alpha=0.8)
ax.bar(x, [fq_cmrv_stats.loc[fq, 'IWM_med'] for fq in fq_order], w,
       label='IWM', color=COLORS['iwm'], alpha=0.7)
ax.bar(x + w, [fq_cmrv_stats.loc[fq, 'SPY_med'] for fq in fq_order], w,
       label='SPY', color=COLORS['spy'], alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([FQ_LABELS[fq] for fq in fq_order], fontsize=9)
ax.set_ylabel('Median Calendar-Month RV (ann %)')
ax.set_title('Median CMRV by Fiscal Quarter: HRB vs Controls')
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
#  18c. Vol Ratio Seasonality — Calendar-Month RV
#       (mirrors Section 10)
# ======================================================================

cmrv_ratio_by_month = cmrv_panel.groupby('month').agg(
    ratio_mean=('HRB_vs_IWM', 'mean'),
    ratio_med=('HRB_vs_IWM', 'median'),
    ratio_p25=('HRB_vs_IWM', lambda x: np.percentile(x, 25)),
    ratio_p75=('HRB_vs_IWM', lambda x: np.percentile(x, 75)),
    n=('HRB_vs_IWM', 'count'),
).round(3)

print("HRB/IWM Calendar-Month Vol Ratio by Month:\n")
print(cmrv_ratio_by_month.to_string())

# --- Chart 1: Median + IQR band ---
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(12)
meds = [cmrv_ratio_by_month.loc[m, 'ratio_med'] for m in range(1, 13)]
p25 = [cmrv_ratio_by_month.loc[m, 'ratio_p25'] for m in range(1, 13)]
p75 = [cmrv_ratio_by_month.loc[m, 'ratio_p75'] for m in range(1, 13)]

ax.fill_between(x, p25, p75, alpha=0.2, color=COLORS['hrb'], label='p25–p75')
ax.plot(x, meds, 'o-', color=COLORS['hrb'], lw=2, markersize=8, label='Median', zorder=5)
ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5, label='Ratio = 1.0')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB CMRV / IWM CMRV')
ax.set_title('HRB/IWM Calendar-Month Vol Ratio Seasonality\n(no overlapping windows)')
ax.legend()
plt.tight_layout()
plt.show()

# --- Chart 2: Strip plot by month (replaces the noisy box plot) ---
fig, ax = plt.subplots(figsize=(14, 7))

for m in range(1, 13):
    vals = cmrv_panel.loc[cmrv_panel.month == m, 'HRB_vs_IWM'].values
    jitter = np.random.default_rng(m + 200).uniform(-0.15, 0.15, size=len(vals))
    ax.scatter(np.full_like(vals, m - 1) + jitter, vals,
               color='#4A90D9', alpha=0.6, s=40, zorder=3, edgecolors='white', linewidth=0.5)
    ax.plot([m - 1 - 0.2, m - 1 + 0.2], [np.median(vals)] * 2,
            color=COLORS['accent'], lw=2.5, zorder=4)

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.axhline(1.0, color=COLORS['neutral'], ls='--', lw=1, alpha=0.5)
ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('HRB CMRV / IWM CMRV')
ax.set_title(f'HRB/IWM calendar-month vol ratio ({START_DATE[:4]}–{END_DATE[:4]})\n'
             f'(each dot = one month, no overlapping data)')
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
#  18d. Year-Over-Year Heatmaps — Calendar-Month RV
#       (mirrors Sections 13 and 14)
# ======================================================================

# --- Heatmap 1: HRB Calendar-Month RV ---
ym_cmrv = cmrv_panel.pivot(index='year', columns='month', values='HRB_cmrv')

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_cmrv) * 0.5)))
vals = ym_cmrv.values
im = ax.imshow(vals, cmap='YlOrRd', aspect='auto',
               vmin=np.nanpercentile(vals, 5), vmax=np.nanpercentile(vals, 95))

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if v > np.nanpercentile(vals, 70) else 'black'
            ax.text(j, i, f'{v:.0f}', ha='center', va='center', fontsize=9, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_cmrv)))
ax.set_yticklabels(ym_cmrv.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title('HRB Calendar-Month RV (ann %) by Year × Month')
plt.colorbar(im, shrink=0.8, label='CMRV (ann %)')
plt.tight_layout()
plt.show()

# --- Heatmap 2: HRB/IWM Vol Ratio ---
ym_cmrv_ratio = cmrv_panel.pivot(index='year', columns='month', values='HRB_vs_IWM')

fig, ax = plt.subplots(figsize=(16, max(6, len(ym_cmrv_ratio) * 0.5)))
vals = ym_cmrv_ratio.values
vmax = max(abs(np.nanpercentile(vals, 5) - 1), abs(np.nanpercentile(vals, 95) - 1))
im = ax.imshow(vals, cmap='RdYlGn_r', aspect='auto',
               vmin=1 - vmax, vmax=1 + vmax)

for i in range(vals.shape[0]):
    for j in range(vals.shape[1]):
        v = vals[i, j]
        if not np.isnan(v):
            color = 'white' if abs(v - 1) > vmax * 0.5 else 'black'
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9, color=color)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_yticks(range(len(ym_cmrv_ratio)))
ax.set_yticklabels(ym_cmrv_ratio.index)
ax.set_xlabel('Calendar Month')
ax.set_ylabel('Year')
ax.set_title('HRB/IWM Calendar-Month Vol Ratio by Year × Month\n(>1 = HRB more volatile than IWM)')
plt.colorbar(im, shrink=0.8, label='HRB CMRV / IWM CMRV')
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================================
#  18e. Statistical Significance — Log Vol Ratio (Calendar-Month RV)
#       (mirrors Section 11, but data is already independent — no
#        overlap correction needed)
# ======================================================================

from scipy import stats as sp_stats

# Grand statistics
cm_grand_mean = cmrv_panel['log_ratio'].mean()
cm_grand_std  = cmrv_panel['log_ratio'].std()
cm_grand_n    = len(cmrv_panel)

# Per-month statistics
cm_lr_stats = pd.DataFrame(index=range(1, 13))

for m in range(1, 13):
    month_vals = cmrv_panel.loc[cmrv_panel.month == m, 'log_ratio']
    n_m   = len(month_vals)
    mean_m = month_vals.mean()
    std_m  = month_vals.std()
    med_m  = month_vals.median()

    cohens_d = (mean_m - cm_grand_mean) / cm_grand_std
    se_m     = std_m / np.sqrt(n_m)
    t_stat   = (mean_m - cm_grand_mean) / se_m
    p_value  = 2 * sp_stats.t.sf(abs(t_stat), df=n_m - 1)
    percentile = sp_stats.percentileofscore(cmrv_panel['log_ratio'].values, med_m)

    cm_lr_stats.loc[m, 'n'] = n_m
    cm_lr_stats.loc[m, 'mean_ratio'] = np.exp(mean_m)
    cm_lr_stats.loc[m, 'med_ratio']  = np.exp(med_m)
    cm_lr_stats.loc[m, 'mean_log']   = mean_m
    cm_lr_stats.loc[m, 'std_log']    = std_m
    cm_lr_stats.loc[m, 'cohens_d']   = cohens_d
    cm_lr_stats.loc[m, 't_stat']     = t_stat
    cm_lr_stats.loc[m, 'p_value']    = p_value
    cm_lr_stats.loc[m, 'percentile'] = percentile

cm_lr_stats['n'] = cm_lr_stats['n'].astype(int)
cm_lr_stats['effect'] = cm_lr_stats['cohens_d'].apply(d_label)
cm_lr_stats['sig'] = cm_lr_stats['p_value'].apply(
    lambda p: '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
)

print(f"Grand mean log(HRB/IWM) [calendar-month RV]: {cm_grand_mean:.4f}  (ratio = {np.exp(cm_grand_mean):.3f})")
print(f"Grand std:               {cm_grand_std:.4f}")
print(f"N year×month cells:      {cm_grand_n}  (~{cm_grand_n//12} years × 12 months)\n")

display_df = cm_lr_stats.copy()
display_df.index = MONTH_NAMES
print(display_df[['n', 'med_ratio', 'mean_log', 'cohens_d', 'effect', 't_stat', 'p_value', 'percentile', 'sig']].round(4).to_string())

print("\nCohen's d: |d|>0.2 small, >0.5 medium, >0.8 large")
print("Significance (t-distribution, n-1 df): *** p<0.01, ** p<0.05, * p<0.1")
print(f"\nEach observation = one calendar month's RV (no overlap). n = {cm_lr_stats['n'].median():.0f} per month.")

In [ ]:
# ======================================================================
#  18f. Visualization — Cohen's d, Percentiles, Strip Plot
#       (mirrors Section 11 charts)
# ======================================================================

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
x = np.arange(12)

# -- Panel 1: Cohen's d
ax = axes[0]
d_vals = [cm_lr_stats.loc[m, 'cohens_d'] for m in range(1, 13)]
bar_colors = []
for d in d_vals:
    if abs(d) >= 0.5:
        bar_colors.append(COLORS['hrb'] if d > 0 else COLORS['pink'])
    elif abs(d) >= 0.2:
        bar_colors.append('#A5D6A7' if d > 0 else '#EF9A9A')
    else:
        bar_colors.append(COLORS['dead_zone'])

ax.bar(x, d_vals, 0.6, color=bar_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(0, color='black', lw=1)
ax.axhline(0.5, color=COLORS['neutral'], ls='--', lw=1, alpha=0.4, label='|d| = 0.5 (medium)')
ax.axhline(-0.5, color=COLORS['neutral'], ls='--', lw=1, alpha=0.4)
ax.axhline(0.2, color=COLORS['neutral'], ls=':', lw=1, alpha=0.3, label='|d| = 0.2 (small)')
ax.axhline(-0.2, color=COLORS['neutral'], ls=':', lw=1, alpha=0.3)

for i, d in enumerate(d_vals):
    ax.text(i, d + (0.03 if d >= 0 else -0.06), f'{d:.2f}', ha='center', fontsize=8, fontweight='bold')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel("Cohen's d")
ax.set_title("Effect Size: Calendar-Month log(HRB/IWM) vs Grand Mean\n(fully independent observations)")
ax.legend(fontsize=8)

# -- Panel 2: Percentile
ax = axes[1]
pct_vals = [cm_lr_stats.loc[m, 'percentile'] for m in range(1, 13)]
pct_colors = [COLORS['hrb'] if p > 60 else (COLORS['pink'] if p < 40 else COLORS['dead_zone']) for p in pct_vals]
ax.bar(x, pct_vals, 0.6, color=pct_colors, alpha=0.8, edgecolor='white', linewidth=0.5)
ax.axhline(50, color='black', ls='--', lw=1, alpha=0.5, label='50th percentile')

for i, p in enumerate(pct_vals):
    ax.text(i, p + 1.5, f'{p:.0f}', ha='center', fontsize=9, fontweight='bold')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(x)
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('Percentile in Overall Distribution')
ax.set_title('Percentile: Calendar-Month log(HRB/IWM)\nvs All Months')
ax.set_ylim(0, 100)
ax.legend(fontsize=8)

# -- Panel 3: Strip plot
ax = axes[2]
for m in range(1, 13):
    month_pts = cmrv_panel.loc[cmrv_panel.month == m, 'log_ratio'].values
    jitter = np.random.default_rng(m + 300).uniform(-0.15, 0.15, size=len(month_pts))
    d = abs(cm_lr_stats.loc[m, 'cohens_d'])
    color = COLORS['hrb'] if d >= 0.2 else COLORS['dead_zone']
    ax.scatter(m - 1 + jitter, month_pts, color=color, alpha=0.6, s=35, zorder=3)
    ax.plot([m - 1 - 0.25, m - 1 + 0.25],
            [month_pts.mean(), month_pts.mean()],
            color=COLORS['accent'], lw=2, zorder=4)

ax.axhline(cm_grand_mean, color=COLORS['pink'], ls='--', lw=1, alpha=0.7,
           label=f'Grand mean = {cm_grand_mean:.3f}')
ax.axhline(0, color=COLORS['neutral'], ls=':', lw=1, alpha=0.3, label='log(ratio) = 0')

for m_idx in [0, 1, 2, 3]:
    ax.axvspan(m_idx - 0.5, m_idx + 0.5, alpha=0.06, color=COLORS['tax_season'], zorder=0)

ax.set_xticks(range(12))
ax.set_xticklabels(MONTH_NAMES)
ax.set_ylabel('log(HRB CMRV / IWM CMRV)')
ax.set_title('Calendar-Month log(HRB/IWM) by Month\n(each dot = one year, bar = month mean)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

# Summary
cm_high = [MONTH_NAMES[m-1] for m in range(1, 13) if cm_lr_stats.loc[m, 'cohens_d'] >= 0.2]
cm_low  = [MONTH_NAMES[m-1] for m in range(1, 13) if cm_lr_stats.loc[m, 'cohens_d'] <= -0.2]
print(f"Months with ELEVATED ratio (d >= 0.2):  {', '.join(cm_high) if cm_high else 'None'}")
print(f"Months with DEPRESSED ratio (d <= -0.2): {', '.join(cm_low) if cm_low else 'None'}")
print(f"\nAll stats use calendar-month RV — fully independent, no rolling-window overlap.")

## 19. Export